### Tier 2 Recommendation Engine (Enhanced)

This notebook trains the second tier of the AyurFit Machine Learning pipeline. It maps the predicted Disease, Symptom Severity, Age, and Gender to specific Ayurvedic Herbs using a transparent Decision Tree algorithm. Age and Gender significantly influence the specific botanical formulation in traditional Ayurvedic practice.

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

### Load Data

We load the curated dataset and explicitly drop any records missing critical information required for the Tier 2 mapping: `Disease`, `Symptom Severity`, or `Ayurvedic Herbs`.

In [2]:
df = pd.read_csv('../dataset/final ayurfit.csv')
df = df.dropna(subset=['Disease', 'Symptom Severity', 'Ayurvedic Herbs'])

### Encode Features and Target

We reuse the exact `disease_label_encoder.joblib` from Tier 1 to maintain architectural consistency. We map `Symptom Severity` to numerical weights. We also parse the raw text in `Age Group` and `Gender` columns to create discrete demographic categories.

In [3]:
# Load Tier 1 Disease Encoder
disease_encoder = joblib.load('disease_label_encoder.joblib')

# Ensure dataset only includes diseases the encoder knows
known_diseases = set(disease_encoder.classes_)
df = df[df['Disease'].isin(known_diseases)]
df['Disease_Encoded'] = disease_encoder.transform(df['Disease'])

# Map Symptom Severity text to integer weights
severity_map = {
    'Low': 0, 'Mild to Moderate': 1, 'Moderate': 2, 'Moderate to High': 3,
    'Moderate to Severe': 4, 'Mild to Severe': 5, 'High': 6, 'Severe': 7
}
df['Severity_Encoded'] = df['Symptom Severity'].map(severity_map).fillna(1).astype(int)

# Map Gender to Categorical (0: Both, 1: Male, 2: Female)
def parse_gender(g):
    g = str(g).lower()
    if 'female' in g or 'women' in g: return 2
    if 'male' in g or 'men' in g: return 1
    return 0

# Map Age Group to Categorical (0: Child, 1: Teen, 2: Adult, 3: Senior)
def parse_age_group(a):
    a = str(a).lower()
    if 'child' in a or 'infant' in a: return 0
    if 'teen' in a or 'adolescent' in a: return 1
    if 'elder' in a or '60+' in a or '80' in a: return 3
    if 'adult' in a or 'age' in a: return 2
    nums = re.findall(r'\d+', a)
    if nums:
        avg = sum(int(n) for n in nums) / len(nums)
        if avg < 13: return 0
        if avg < 20: return 1
        if avg < 60: return 2
        return 3
    return 2

df['Gender_Encoded'] = df['Gender'].apply(parse_gender)
df['Age_Encoded'] = df['Age Group'].apply(parse_age_group)

# Encode the target Ayurvedic Herbs
herbs_encoder = LabelEncoder()
df['Herbs_Encoded'] = herbs_encoder.fit_transform(df['Ayurvedic Herbs'])

### Train/Test Split

We isolate our enhanced 4-dimensional input feature matrix `X` and our target matrix `y` (Herbs), splitting 80% for training and 20% for evaluation.

In [4]:
X = df[['Disease_Encoded', 'Severity_Encoded', 'Age_Encoded', 'Gender_Encoded']]
y = df['Herbs_Encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Train Decision Tree

Using a Decision Tree ensures explicit, explainable clinical rules for remedy mapping. It naturally captures the logical hierarchy between the primary disease, severity, and critical demographic factors.

In [5]:
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train)

train_acc = accuracy_score(y_train, dt_classifier.predict(X_train))
test_acc = accuracy_score(y_test, dt_classifier.predict(X_test))

print(f"Training Accuracy: {train_acc * 100:.2f}%")
print(f"Testing Accuracy: {test_acc * 100:.2f}%")

Training Accuracy: 95.75%
Testing Accuracy: 88.26%


### Export Models

We export the enhanced Tier 2 mapping model and the `herbs_label_encoder.joblib` for the production backend.

In [6]:
joblib.dump(dt_classifier, './dis-rem imp files/tier2_recommender.joblib')
joblib.dump(herbs_encoder, './dis-rem imp files/herbs_label_encoder.joblib')
print("Tier 2 Models successfully exported!")

Tier 2 Models successfully exported!
